In [1]:
import pandas as pd

In [2]:
df_checks = pd.read_excel("./Analise manual Aos fatos.xlsx")


In [3]:
df_parls = pd.read_csv("../../data/df_parlamentares_por_legislatura.csv")
df_parls = df_parls[df_parls['idLegislatura'].isin([56, 57])][['id', 'nome']].drop_duplicates()

In [93]:
df_ifs = (
    df_checks.groupby(["Parlamentar", "Partido", "Ano"])
    .agg({"Link": "count"})
    .reset_index()
    .sort_values("Link", ascending=False)
    .rename(columns={"Link": "qtde"})
)

In [95]:
def treat_name(name):
    depara = {
        'Maurício Marcon': 'Marcon',
        " André Janones": "André Janones",
        "Cabo Gilberto": "Cabo Gilberto Silva",
        "Felipe Barros": "Filipe Barros",
        "Eliéser Girão": "General Girão",
        "Marcel Van Hattem": "Marcel van Hattem",
        "Paulo Bilynskyj": "Delegado Paulo Bilynskyj"
    }
    return depara.get(name, name)

df_ifs['ifs'] = df_ifs['qtde'].apply(lambda x: max(1 - x *.2, 0))

df_ifs['nome'] = df_ifs['Parlamentar'].apply(treat_name)
df_ifs = df_ifs.merge(df_parls, left_on='nome', right_on="nome", how='left')
del df_ifs['Parlamentar']

In [123]:
d = (
    df_ifs.groupby("Ano")
    .agg({"ifs": "mean", "qtde": "sum", "nome": "count"})
    .reset_index()
)

d.rename(
    columns={
        "ifs": "IFS médio",
        "qtde": "Total de publicações",
        "nome": "Quantidade de parlamentares",
    },
    inplace=True,
)
display(d)

d.to_latex("../../docs/tables/ifs_parls.tex", index=False, float_format="%.1f")

,Ano,IFS médio,Total de publicações,Quantidade de parlamentares
0,2023,0.700000,12,8
1,2024,0.707692,19,13


In [122]:
d = df_ifs.groupby("Partido").agg({"nome": "nunique", "qtde": "sum"}).reset_index()

d["%_do_total"] = (d["nome"] / d["nome"].sum() * 100).round(1)

d.sort_values("%_do_total", ascending=False, inplace=True)

d.rename(
    columns={
        "nome": "Quantidade de parlamentares",
        "qtde": "Total de publicações",
        "%_do_total": r"\% de parlamentares",
    },
    inplace=True,
)
display(d)

d.to_latex("../../docs/tables/ifs_partidos.tex", index=False, float_format="%.1f")

,Partido,Quantidade de parlamentares,Total de publicações,\% de parlamentares
3,PL,9,17,50.0
5,PSOL,2,3,11.1
0,Avante,1,2,5.6
2,Novo,1,1,5.6
1,MDB,1,1,5.6
4,PSB,1,1,5.6
6,Podemos,1,4,5.6
7,Repúblicanos,1,1,5.6
8,União Brasil,1,1,5.6


In [102]:
df_ifs

,Partido,Ano,qtde,ifs,nome,id
0,Podemos,2024,4,0.2,Marcon,160535
1,PL,2024,3,0.4,Gustavo Gayer,220568
2,PL,2023,3,0.4,Gustavo Gayer,220568
3,PL,2023,2,0.6,Bia Kicis,204374
4,Avante,2023,2,0.6,André Janones,204515
5,PSOL,2024,2,0.6,Guilherme Boulos,220639
6,PL,2023,1,0.8,Cabo Gilberto Silva,220574
7,PL,2024,1,0.8,Bia Kicis,204374
8,PL,2023,1,0.8,Bibo Nunes,204388
9,PL,2024,1,0.8,Filipe Barros,204411
